In [ ]:
# ── CELL 1 — Constants + install dependencies ──────────────────────────────
#
# Branch: ctclip-stage2 — DO NOT MERGE TO MAIN.
#
# Extracts CT-CLIP visual features for the full CT-RATE dataset and uploads
# them to Google Drive via rclone. Fully resumable: a crash, disconnect or
# OOM loses at most FLUSH_EVERY volumes of work.
#
# Model init and preprocessing are copied verbatim from
# ctclip_feasibility_test.py in this repo - do not rewrite them here.
#
# COLAB SETUP (do these first):
#   1. Runtime -> Change runtime type -> T4 GPU.
#   2. Add HF_TOKEN as a notebook Secret (key icon in the left sidebar,
#      'Notebook access' toggled on). It must be a token that has accepted
#      the CT-RATE gated-dataset terms on HuggingFace.
#   3. Paste your rclone.conf into RCLONE_CONF in CELL 2.
#   4. Run CELL 1, RESTART THE RUNTIME, then run CELL 1-7 in order.
#   5. Set RCLONE_REMOTE below to match your rclone remote name.
#
# SCALE WARNING: the full dataset is ~50,188 volumes (47,149 train + 3,039
# valid). Extracting all of it means DOWNLOADING ~7.8 TB of raw CT from
# HuggingFace to produce ~677 GB of features. The download - not the GPU - is
# the bottleneck, and this will span many sessions. That's why every part of
# this notebook is resumable.

import os
from pathlib import Path

# ---- User-tunable constants ----
RCLONE_REMOTE = "driveB:"      # UPDATE to match your configured rclone remote

# Colab doesn't populate env vars, so read the notebook Secret first and fall
# back to a real env var when running outside Colab.
try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
except Exception:
    pass
HF_TOKEN         = os.environ.get("HF_TOKEN")
STAGING_DIR      = "/content/staging"
BATCH_SIZE       = 8            # reduce to 2 or 1 if VRAM OOM
DOWNLOAD_WORKERS = 8            # concurrent downloads per batch - benchmarked
DOWNLOAD_RETRIES = 5            # attempts per volume before giving up. HF resets
                                # connections under sustained concurrent load; without
                                # retries a single blip kills the whole loop and loses
                                # everything since the last flush.
PREPROCESS_WORKERS = 4          # concurrent nibabel loads per batch. This is the
                                # slowest stage (~2.7s/vol serial): gzip decompress
                                # + numpy crop/pad. Threads (not processes) because
                                # zlib and numpy both release the GIL, and processes
                                # would have to pickle ~221MB per volume back.
                                # Set to 1 to go back to serial.
                                 # ~3x faster than sequential on Colab's
                                 # network, plateauing past ~4-8 workers
                                 # (see benchmark_download_speed.ipynb)
FLUSH_EVERY      = 500         # rclone upload cadence (volumes, not batches)
LOG_EVERY        = 20           # progress-line cadence (volumes)
TARGET_SHAPE     = (480, 480, 240)

# ---- PARALLEL SHARDING (optional: run two Colab accounts at once) ----
# Split the dataset across accounts by a stable hash of the volume name, so
# each account processes a DISJOINT slice and they never download the same
# volume. No runtime coordination between accounts is needed - the split is
# decided identically on both machines by the hash.
#
#   Account A:  SHARD_INDEX = 0,  SHARD_COUNT = 2
#   Account B:  SHARD_INDEX = 1,  SHARD_COUNT = 2
#
# BOTH accounts MUST use the SAME SHARD_COUNT and DIFFERENT SHARD_INDEX
# (each in 0..SHARD_COUNT-1). Setting both to the same index just makes them
# redo each other's work - wasteful, but not corrupting (identical volume ->
# identical feature file). SHARD_COUNT = 1 is the original single-account
# behaviour (no split).
#
# The ~6,800 volumes already done under the old single-account run are NOT
# re-extracted: CELL 4 loads every checkpoint file (including the legacy
# checkpoint.json) and both accounts skip anything already done.
SHARD_INDEX = 0
SHARD_COUNT = 2
assert 0 <= SHARD_INDEX < SHARD_COUNT, "Need 0 <= SHARD_INDEX < SHARD_COUNT."

# RCLONE_REMOTE already ends in ':', so remote paths concatenate straight onto
# it - do NOT add a second colon.
REMOTE_ROOT       = RCLONE_REMOTE + "ctrate_features"

# Per-shard checkpoint + log filenames, so two accounts on different shards
# never overwrite each other. CELL 4 still LOADS every checkpoint*.json on
# resume (this shard, the other, and the legacy checkpoint.json), so each
# account sees all completed work; it only ever WRITES its own file.
SHARD_TAG         = f"shard{SHARD_INDEX}of{SHARD_COUNT}"
MY_CKPT_NAME      = f"checkpoint_{SHARD_TAG}.json"
REMOTE_CHECKPOINT = REMOTE_ROOT + "/" + MY_CKPT_NAME
LOCAL_CHECKPOINT  = Path("/content/" + MY_CKPT_NAME)
LOG_FILE          = Path(f"/content/extract_log_{SHARD_TAG}.txt")

HF_REPO = "ibrahimhamamci/CT-RATE"
# Real CT-RATE v2 layout, verified empirically in this project:
#   dataset/train_fixed/{patient}/{scan}/{volume}.nii.gz
#   e.g. dataset/train_fixed/train_1/train_1_a/train_1_a_1.nii.gz
SPLITS = {"train": "dataset/train_fixed", "valid": "dataset/valid_fixed"}

CT_CLIP_REPO_URL  = "https://github.com/ibrahimethemhamamci/CT-CLIP.git"
CT_CLIP_LOCAL_DIR = Path("./CT-CLIP")
WORK_DIR     = Path("./ctclip_work")
WEIGHTS_PATH = WORK_DIR / "CT-CLIP_v2.pt"

# CTViT / CTCLIP construction config, from CT-CLIP's scripts/run_zero_shot.py.
CTVIT_CONFIG = dict(
    dim=512,
    codebook_size=8192,
    image_size=480,
    patch_size=20,
    temporal_patch_size=10,
    spatial_depth=4,
    temporal_depth=4,
    dim_head=32,
    heads=8,
)
CTCLIP_CONFIG = dict(
    dim_image=294912,
    dim_text=768,
    dim_latent=512,
    extra_latent_projection=False,
    use_mlm=False,
    downsample_image_embeds=False,
    use_all_token_embeds=False,
)
TEXT_ENCODER_NAME = "microsoft/BiomedVLP-CXR-BERT-specialized"

assert HF_TOKEN, (
    "No HF_TOKEN. On Colab: add it as a notebook Secret (key icon, left "
    "sidebar) with 'Notebook access' on. Elsewhere: export HF_TOKEN=... . "
    "It must have accepted the CT-RATE gated-dataset terms."
)

WORK_DIR.mkdir(parents=True, exist_ok=True)
Path(STAGING_DIR).mkdir(parents=True, exist_ok=True)

# ---- Install dependencies ----
import subprocess
import sys


def _pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


_pip_install("nibabel", "huggingface_hub", "transformers", "torch")

if not CT_CLIP_LOCAL_DIR.exists():
    subprocess.run(["git", "clone", CT_CLIP_REPO_URL, str(CT_CLIP_LOCAL_DIR)], check=True)
else:
    print(f"{CT_CLIP_LOCAL_DIR} already exists, skipping clone.")

# CT-CLIP has NO setup.py at its repo root: transformer_maskgit (CTViT) and
# CT_CLIP (the CTCLIP wrapper) are each their own installable subdirectory,
# each nesting the real package one level deeper. Do NOT add the repo root to
# sys.path - the outer transformer_maskgit/ folder has no __init__.py and would
# shadow the real install as an empty namespace package.
for sub in ("transformer_maskgit", "CT_CLIP"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(CT_CLIP_LOCAL_DIR / sub)],
        check=True,
    )

print("Dependencies installed.")
print("If this was a fresh install, RESTART THE RUNTIME now, then re-run CELL 1")
print("before continuing - a failed import caches a broken module in sys.modules.")

In [ ]:
# ── CELL 2 — rclone config ─────────────────────────────────────────────────

import pathlib
import shutil as _shutil
import subprocess

# Colab images don't ship rclone - install it if it isn't already present.
if _shutil.which("rclone") is None:
    print("Installing rclone ...")
    subprocess.run(
        "curl -s https://rclone.org/install.sh | sudo bash",
        shell=True, check=True,
    )
else:
    print("rclone already installed.")

# ===========================================================================
# TODO: PASTE YOUR rclone.conf VALUES BELOW.
#
# Get them by running `rclone config show` on a machine where the remote
# already works. It should look like:
#
#   [driveB]
#   type = drive
#   client_id = ....apps.googleusercontent.com
#   client_secret = GOCSPX-...
#   scope = drive
#   token = {"access_token":"...","refresh_token":"...","expiry":"..."}
#
# The section name in brackets must match RCLONE_REMOTE from CELL 1 (minus the
# trailing colon) - i.e. "driveB" for RCLONE_REMOTE = "driveB:".
#
# CLIENT_ID / CLIENT_SECRET must be YOUR OWN rclone Drive app credentials
# (Google Cloud Console -> APIs & Services -> Credentials), not rclone's
# shared one. The shared client_id has a much lower per-user API quota and
# throttles hard under sustained traffic like this notebook's flushes -
# that's what was causing multi-minute FLUSH steps. A token minted under the
# old shared client_id will NOT work here - RCLONE_TOKEN must be a fresh
# token obtained under your own CLIENT_ID/CLIENT_SECRET.
#
# SECURITY: these values are live OAuth credentials. Do NOT commit this
# notebook with them filled in, and do not share the executed notebook.
# ===========================================================================

import pathlib

CLIENT_ID     = "YOUR_CLIENT_ID.apps.googleusercontent.com"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
RCLONE_TOKEN  = '{"access_token":"YOUR_OAUTH_ACCESS_TOKEN","token_type":"Bearer","refresh_token":"YOUR_OAUTH_REFRESH_TOKEN","expiry":"2026-07-09T21:41:40.6915615+02:00","expires_in":3599}'


!curl -s https://rclone.org/install.sh | sudo bash
conf = f'''[driveB]
type = drive
client_id = {CLIENT_ID}
client_secret = {CLIENT_SECRET}
scope = drive
token = {RCLONE_TOKEN}
'''
cfg = pathlib.Path.home() / ".config/rclone/rclone.conf"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text(conf)
!rclone lsd driveB: | head     # verify the remote (lists B's folders)


# RCLONE_CONF = """{"access_token":"YOUR_OAUTH_ACCESS_TOKEN","token_type":"Bearer","refresh_token":"YOUR_OAUTH_REFRESH_TOKEN","expiry":"2026-07-06T12:02:01.1962304+02:00","expires_in":3599}"""  # <-- PASTE BETWEEN THE TRIPLE QUOTES

# assert RCLONE_CONF.strip(), "Paste your rclone.conf content into RCLONE_CONF above."

# cfg = pathlib.Path.home() / ".config/rclone/rclone.conf"
# cfg.parent.mkdir(parents=True, exist_ok=True)
# cfg.write_text(RCLONE_CONF)
# print(f"Wrote {cfg}")

# # Verify the remote actually resolves before spending hours extracting.
# !rclone lsd {RCLONE_REMOTE} | head


In [ ]:
import pathlib
p = pathlib.Path("/root/.config/rclone/rclone.conf")
print("exists:", p.exists(), "| bytes:", p.stat().st_size if p.exists() else 0)
print("sections found:", [l.strip() for l in p.read_text().splitlines() if l.strip().startswith("[")])


In [ ]:
# ── CELL 3 — Build and load CT-CLIP ────────────────────────────────────────
# Copied verbatim from ctclip_feasibility_test.py.

# ---- Download CT-CLIP_v2.pt weights ----
if WEIGHTS_PATH.exists():
    print(f"Weights already downloaded at {WEIGHTS_PATH}, skipping.")
else:
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    all_files = api.list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
    candidates = [f for f in all_files if "CT-CLIP_v2" in f and f.endswith(".pt")]
    if not candidates:
        raise FileNotFoundError(
            f"No file matching 'CT-CLIP_v2*.pt' found in {HF_REPO}."
        )
    remote_path = candidates[0]
    print(f"Found weights at: {remote_path}")

    local_path = Path(
        hf_hub_download(
            repo_id=HF_REPO,
            repo_type="dataset",
            filename=remote_path,
            token=HF_TOKEN,
            local_dir=str(WORK_DIR),
        )
    )
    if local_path != WEIGHTS_PATH:
        local_path.rename(WEIGHTS_PATH)
print(f"Weights ready at {WEIGHTS_PATH}")

# ---- Build model ----
import torch
from transformer_maskgit import CTViT
from transformers import BertModel, BertTokenizer

try:
    from ct_clip import CTCLIP
except ImportError as exc:
    raise ImportError(
        "Could not import CTCLIP from the ct_clip package - check "
        "CT-CLIP/scripts/run_zero_shot.py in the cloned repo for the current "
        "correct import path if the package layout has changed."
    ) from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

text_tokenizer = BertTokenizer.from_pretrained(TEXT_ENCODER_NAME)
text_encoder = BertModel.from_pretrained(TEXT_ENCODER_NAME)
# Text encoder is required for CTCLIP's __init__ but not used at inference -
# only clip.visual_transformer(...) is called below.

image_encoder = CTViT(**CTVIT_CONFIG)

clip = CTCLIP(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    tokenizer=text_tokenizer,
    **CTCLIP_CONFIG,
)

# Don't use clip.load(): it does a strict load_state_dict, which rejects the
# whole checkpoint over version drift. transformers >=4.31 removed the
# `position_ids` buffer from BertModel's embeddings, but CT-CLIP_v2.pt was
# saved when it still existed. It's a non-learned arange() buffer, so dropping
# it loses nothing.
state_dict = torch.load(str(WEIGHTS_PATH), map_location="cpu")
state_dict.pop("text_transformer.embeddings.position_ids", None)

# Load non-strictly, but REPORT what didn't match. A silently-missing
# visual_transformer.* key would leave that part of the encoder randomly
# initialised and make every extracted feature meaningless.
missing, unexpected = clip.load_state_dict(state_dict, strict=False)
if missing:
    print(f"WARNING - {len(missing)} missing key(s), first few: {missing[:5]}")
if unexpected:
    print(f"WARNING - {len(unexpected)} unexpected key(s), first few: {unexpected[:5]}")
if not missing and not unexpected:
    print("All checkpoint keys matched exactly.")

visual_missing = [k for k in missing if k.startswith("visual_transformer")]
if visual_missing:
    raise RuntimeError(
        f"{len(visual_missing)} visual_transformer key(s) missing from the "
        f"checkpoint (e.g. {visual_missing[:3]}). The visual encoder would be "
        "partly random and every extracted feature meaningless - fix this "
        "before trusting any output from this notebook."
    )

clip = clip.to(DEVICE)
clip.eval()
for p in clip.parameters():
    p.requires_grad_(False)
print("CTCLIP model built and weights loaded.")

# ---- Preprocessing (copied verbatim from ctclip_feasibility_test.py) ----
import nibabel as nib
import numpy as np


def center_crop_or_pad(arr: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    """Crop (centered) or zero-pad (centered) each axis independently to
    reach target_shape. Operates on an (H, W, D)-ordered array."""
    out = arr
    for axis, target in enumerate(target_shape):
        cur = out.shape[axis]
        if cur > target:
            start = (cur - target) // 2
            idx = [slice(None)] * out.ndim
            idx[axis] = slice(start, start + target)
            out = out[tuple(idx)]
        elif cur < target:
            pad_total = target - cur
            pad_before = pad_total // 2
            pad_after = pad_total - pad_before
            pad_width = [(0, 0)] * out.ndim
            pad_width[axis] = (pad_before, pad_after)
            out = np.pad(out, pad_width, mode="constant", constant_values=0.0)
    return out


def load_and_preprocess(path: Path) -> torch.Tensor:
    img = nib.load(str(path))
    arr = np.asarray(img.dataobj, dtype=np.float32)  # raw NIfTI axis order: (H, W, D)

    arr = np.clip(arr, -1000.0, 1000.0) / 1000.0  # HU clip -> [-1, 1]
    arr = center_crop_or_pad(arr, TARGET_SHAPE)  # (480, 480, 240)

    arr = arr.transpose(2, 0, 1)  # (H, W, D) -> (D, H, W)
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # (1, 1, 240, 480, 480)
    return tensor.float()


In [ ]:
# ── CELL 4 — Resume checkpoint + logging ───────────────────────────────────

import json
import subprocess
import time


def log(msg: str) -> None:
    """Print AND append to a persistent log file. The log is uploaded next to
    the checkpoint on every flush, so history survives a session dying."""
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as fh:
        fh.write(line + "\n")


# Pull EVERY checkpoint file - the legacy single-account checkpoint.json AND
# both per-shard files - and union them into `done`. This is what lets two
# accounts cooperate with zero coordination: each sees everything already
# finished (by itself, the other shard, or the old unsharded run) and skips
# it, while CELL 5 restricts NEW work to this shard.
CKPT_DIR = Path("/content/ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["rclone", "copy", REMOTE_ROOT, str(CKPT_DIR),
     "--include", "checkpoint*.json", "--ignore-errors"],
    check=False,
)
# Pull this shard's own log so we append to its running history.
subprocess.run(
    ["rclone", "copy", REMOTE_ROOT + "/" + LOG_FILE.name, "/content/", "--ignore-errors"],
    check=False,
)

done = set()
ckpt_files = sorted(CKPT_DIR.glob("checkpoint*.json"))
for cf in ckpt_files:
    try:
        done |= set(json.loads(cf.read_text()).get("done", []))
    except Exception as exc:  # noqa: BLE001 - a partial file must not abort resume
        log(f"RESUME: skipping unreadable {cf.name}: {exc}")
if ckpt_files:
    log(f"RESUME: unioned {len(ckpt_files)} checkpoint file(s) -> "
        f"{len(done):,} volume(s) already done (all shards).")
else:
    log("RESUME: no checkpoints on the remote - starting fresh.")


def save_checkpoint(done_set: set) -> None:
    """Write THIS shard's checkpoint + log and mirror both to the remote.
    Only ever writes MY_CKPT_NAME, so a parallel account writing its own
    shard file can't be clobbered."""
    LOCAL_CHECKPOINT.write_text(json.dumps({"done": sorted(done_set)}))
    subprocess.run(["rclone", "copy", str(LOCAL_CHECKPOINT), REMOTE_ROOT + "/"], check=False)
    if LOG_FILE.exists():
        subprocess.run(["rclone", "copy", str(LOG_FILE), REMOTE_ROOT + "/"], check=False)


In [ ]:
# ── CELL 5 — Volume discovery ──────────────────────────────────────────────

import zlib

from huggingface_hub import HfApi

api = HfApi()

# One recursive tree listing per split, rather than walking ~20k directories
# with individual ls calls. Metadata-only (no downloads) - the same approach
# scripts/check_dataset_size.py uses in this repo. `size` comes back with each
# entry, so we get the real raw-byte total for free.
# Carry `size` through: passing it to fs.open() in CELL 6 skips fsspec's
# per-file info()/ls() API round-trip (it only needs the size to set up a
# read cache we never use, since we read each file whole).
all_volumes = []  # list of (split, hf_repo_path, volname, size_bytes)
raw_bytes_all = 0
for split, root in SPLITS.items():
    print(f"Listing {root} ...")
    n_split = 0
    for entry in api.list_repo_tree(
        HF_REPO, repo_type="dataset", path_in_repo=root,
        recursive=True, token=HF_TOKEN,
    ):
        path = getattr(entry, "path", "")
        if path.endswith(".nii.gz"):
            volname = Path(path).name[: -len(".nii.gz")]
            size_bytes = getattr(entry, "size", 0) or 0
            all_volumes.append((split, path, volname, size_bytes))
            raw_bytes_all += size_bytes
            n_split += 1
    log(f"DISCOVERY: {split:5s} -> {n_split:,} volume(s)")

# Which volumes belong to THIS shard (regardless of done-status). crc32
# (NOT Python's hash(), which is per-process randomised) so both accounts
# compute the identical, disjoint split on every run.
def _is_mine(volname: str) -> bool:
    return SHARD_COUNT == 1 or zlib.crc32(volname.encode()) % SHARD_COUNT == SHARD_INDEX

my_shard_all = [v for v in all_volumes if _is_mine(v[2])]
MY_SHARD_TOTAL = len(my_shard_all)          # all volumes in my shard
remaining = [v for v in my_shard_all if v[2] not in done]
MY_SHARD_DONE_START = MY_SHARD_TOTAL - len(remaining)  # my shard already done

if SHARD_COUNT > 1:
    log(f"SHARD: index {SHARD_INDEX}/{SHARD_COUNT} - {MY_SHARD_TOTAL:,} volumes in "
        f"my shard, {MY_SHARD_DONE_START:,} already done, {len(remaining):,} to go.")

raw_gb_all = raw_bytes_all / 1e9
pct_done = (100 * len(done) / len(all_volumes)) if all_volumes else 0.0

log(f"DISCOVERY: total={len(all_volumes):,} done={len(done):,} ({pct_done:.1f}%) "
    f"remaining={len(remaining):,}")
log(f"DISCOVERY: raw dataset size = {raw_gb_all:.1f} GB "
    f"({raw_gb_all / max(len(all_volumes), 1) * 1000:.0f} MB/volume avg) - this is "
    f"how much must be downloaded from HF in total across all sessions.")


In [ ]:
# ── CELL 6 — Batched extraction loop ───────────────────────────────────────

import shutil
import threading
from concurrent.futures import ThreadPoolExecutor

from huggingface_hub import HfFileSystem

# HfFileSystem wraps a single shared httpx.Client - one thread's `with
# fs.open(...)` exiting can close that client mid-request for another thread
# ("Cannot send a request, as the client has been closed"). Give each worker
# thread its own instance instead of sharing one across the pool.
_fs_local = threading.local()


def _thread_fs() -> HfFileSystem:
    if not hasattr(_fs_local, "fs"):
        _fs_local.fs = HfFileSystem(token=HF_TOKEN)
    return _fs_local.fs


# Session counters
uploaded_bytes = 0          # bytes actually pushed to the remote this session
feature_bytes_total = 0     # bytes of .pt features produced this session
raw_bytes_downloaded = 0    # bytes of raw NIfTI pulled from HF this session
processed_this_session = 0
since_flush = 0
since_log = 0
dl_seconds = 0.0
pre_seconds = 0.0
enc_seconds = 0.0
oom_fallbacks = 0
t_start = time.time()


def flush_staging() -> tuple:
    """Upload everything staged so far, clear staging, persist checkpoint+log.
    Returns (n_files, gb_uploaded)."""
    global uploaded_bytes
    staged = list(Path(STAGING_DIR).rglob("*.pt"))
    n, gb = len(staged), 0.0
    if staged:
        batch_bytes = sum(f.stat().st_size for f in staged)
        uploaded_bytes += batch_bytes
        gb = batch_bytes / 1e9
        subprocess.run(
            ["rclone", "copy", STAGING_DIR, REMOTE_ROOT + "/", "--progress"],
            check=True,
        )
        for split_dir in Path(STAGING_DIR).iterdir():
            if split_dir.is_dir():
                shutil.rmtree(split_dir)
    save_checkpoint(done)
    return n, gb


def download_one(hf_path: str, out_path: Path, size_bytes: int) -> int:
    """Download a single NIfTI. Runs in a worker thread - downloads are
    I/O-bound waiting on HF's response, so threads pipeline through that wait
    instead of the batch paying it one file at a time. Benchmarked ~3x faster
    than sequential, plateauing past ~4-8 workers on Colab's network (see
    benchmark_download_speed.ipynb). Uses a thread-local HfFileSystem, since
    a shared instance's underlying httpx.Client isn't safe across threads.

    size_bytes is passed straight through to fs.open() so fsspec skips its
    per-file info()/ls() API call - it only wants the size to size a read
    cache for range requests, and we read each file whole in one go. CELL 5
    already knows every size from list_repo_tree, so that round-trip is pure
    overhead (and extra load on HF, which resets connections under it)."""
    last_exc = None
    for attempt in range(DOWNLOAD_RETRIES):
        try:
            fs = _thread_fs()
            with fs.open(
                f"datasets/{HF_REPO}/{hf_path}", "rb", size=size_bytes
            ) as src, open(out_path, "wb") as dst:
                data = src.read()
                dst.write(data)
                return len(data)
        except Exception as exc:  # noqa: BLE001 - any transport error is retryable here
            last_exc = exc
            # Drop this thread's client: after a reset its httpx.Client and the
            # fs dircache may both be in a bad state, and reusing them tends to
            # fail again immediately.
            if hasattr(_fs_local, "fs"):
                del _fs_local.fs
            if attempt < DOWNLOAD_RETRIES - 1:
                wait = 2 ** attempt  # 1, 2, 4, 8 ...
                log(f"WARNING: download failed for {Path(hf_path).name} "
                    f"(attempt {attempt + 1}/{DOWNLOAD_RETRIES}): "
                    f"{type(exc).__name__}: {exc} - retrying in {wait}s")
                time.sleep(wait)
    raise RuntimeError(
        f"Download failed after {DOWNLOAD_RETRIES} attempts: {hf_path}"
    ) from last_exc


def encode(tensor):
    """Encode (B, 1, 240, 480, 480) -> (B, 24, 24, 24, 512)."""
    with torch.no_grad():
        return clip.visual_transformer(tensor.to(DEVICE), return_encoded_tokens=True)


batches = [remaining[i : i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
log(f"START: {len(remaining):,} volume(s) to go, in {len(batches):,} batch(es) "
    f"of up to {BATCH_SIZE}. Flushing every {FLUSH_EVERY} volumes.")

for batch in batches:
    # (a) download each NIfTI in the batch concurrently
    t0 = time.perf_counter()
    tmp_paths = [Path(f"/content/tmp_{i}.nii.gz") for i in range(len(batch))]
    with ThreadPoolExecutor(max_workers=min(DOWNLOAD_WORKERS, len(batch))) as ex:
        futures = [
            ex.submit(download_one, hf_path, tmp, size_bytes)
            for (split, hf_path, volname, size_bytes), tmp in zip(batch, tmp_paths)
        ]
        raw_bytes_downloaded += sum(f.result() for f in futures)
    dl_seconds += time.perf_counter() - t0

    # (b) preprocess each -> list of (1, 1, 240, 480, 480), concurrently.
    # ex.map preserves input order, so `tensors` stays aligned with `batch` -
    # critical, since step (f) pairs features[i] with batch[i] to name the file.
    t0 = time.perf_counter()
    if PREPROCESS_WORKERS > 1 and len(batch) > 1:
        with ThreadPoolExecutor(
            max_workers=min(PREPROCESS_WORKERS, len(batch))
        ) as ex:
            tensors = list(ex.map(load_and_preprocess, tmp_paths))
    else:
        tensors = [load_and_preprocess(p) for p in tmp_paths]
    # (c) each tensor already carries a leading batch dim, so cat (not stack)
    # to get (B, 1, 240, 480, 480) - stack would add a spurious 6th dim.
    batch_tensor = torch.cat(tensors, dim=0)
    pre_seconds += time.perf_counter() - t0

    # (d)/(e) encode, falling back to one-by-one on OOM
    t0 = time.perf_counter()
    try:
        features = encode(batch_tensor).cpu()
    except torch.cuda.OutOfMemoryError:
        oom_fallbacks += 1
        log(f"WARNING: CUDA OOM at BATCH_SIZE={len(batch)} - retrying this batch "
            f"one volume at a time. Lower the BATCH_SIZE constant in CELL 1 to "
            f"stop paying this retry cost on every batch. "
            f"(OOM fallbacks so far: {oom_fallbacks})")
        del batch_tensor
        torch.cuda.empty_cache()
        feats = []
        for t in tensors:
            feats.append(encode(t).cpu())
            torch.cuda.empty_cache()
        features = torch.cat(feats, dim=0)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    enc_seconds += time.perf_counter() - t0

    # (f) save each feature under its ORIGINAL volume name, so training-time
    # code can find it from the reports CSV VolumeName with a plain
    # volname + ".pt" lookup. Never index numbers.
    for i, (split, hf_path, volname, size_bytes) in enumerate(batch):
        out_path = Path(STAGING_DIR) / split / f"{volname}.pt"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(features[i].half(), out_path)
        feature_bytes_total += out_path.stat().st_size

    # (g) drop the temp NIfTIs immediately - they're ~119-156 MB each
    for tmp in tmp_paths:
        tmp.unlink(missing_ok=True)
    del features, tensors
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # (h) mark done
    for split, hf_path, volname, size_bytes in batch:
        done.add(volname)
    processed_this_session += len(batch)
    since_flush += len(batch)
    since_log += len(batch)

    # progress line
    if since_log >= LOG_EVERY:
        since_log = 0
        elapsed = time.time() - t_start
        rate = processed_this_session / elapsed if elapsed > 0 else 0.0
        # Progress + ETA are against MY shard, not the global dataset: this
        # account only does its shard, and it can't see the other account's
        # concurrent work. my_done/eta are exact; the global figure is a
        # lower bound (>=) since it omits the other shard's live progress.
        my_done = MY_SHARD_DONE_START + processed_this_session
        my_left = MY_SHARD_TOTAL - my_done
        eta_h = (my_left / rate / 3600) if rate > 0 else float("nan")
        free_gb = shutil.disk_usage("/content").free / 1e9
        log(
            f"[shard {SHARD_INDEX}/{SHARD_COUNT} {my_done:,}/{MY_SHARD_TOTAL:,} "
            f"{100 * my_done / MY_SHARD_TOTAL:4.1f}%] "
            f"global>={len(done):,}/{len(all_volumes):,} | "
            f"feat={feature_bytes_total / 1e9:.2f}GB up={uploaded_bytes / 1e9:.2f}GB "
            f"raw_dl={raw_bytes_downloaded / 1e9:.1f}GB | "
            f"dl={dl_seconds / processed_this_session:.1f}s "
            f"pre={pre_seconds / processed_this_session:.1f}s "
            f"enc={enc_seconds / processed_this_session:.1f}s /vol | "
            f"{rate * 3600:.0f}vol/h | elapsed={elapsed / 3600:.2f}h ETA={eta_h:.1f}h | "
            f"disk={free_gb:.0f}GB free"
        )

    # (i) flush on a volume (not batch) cadence
    if since_flush >= FLUSH_EVERY:
        since_flush = 0
        n_files, gb = flush_staging()
        log(f"FLUSH: uploaded {gb:.2f}GB ({n_files} file(s)) | "
            f"session total {uploaded_bytes / 1e9:.2f}GB | "
            f"checkpoint saved at {len(done):,} done")

In [ ]:
# ── CELL 7 — Final flush + summary ─────────────────────────────────────────

n_files, gb = flush_staging()
log(f"FINAL FLUSH: uploaded {gb:.2f}GB ({n_files} file(s))")

elapsed = time.time() - t_start
log("=" * 70)
log("SESSION COMPLETE")
log(f"  volumes processed this session: {processed_this_session:,}")
log(f"  this shard done:                {MY_SHARD_DONE_START + processed_this_session:,} / {MY_SHARD_TOTAL:,} "
    f"({100 * (MY_SHARD_DONE_START + processed_this_session) / MY_SHARD_TOTAL:.1f}%)")
log(f"  this shard remaining:           {MY_SHARD_TOTAL - (MY_SHARD_DONE_START + processed_this_session):,}")
log(f"  global done (>= this session):  {len(done):,} / {len(all_volumes):,}")
log(f"  features produced this session: {feature_bytes_total / 1e9:.2f} GB")
log(f"  uploaded this session:          {uploaded_bytes / 1e9:.2f} GB")
log(f"  raw downloaded this session:    {raw_bytes_downloaded / 1e9:.1f} GB")
log(f"  wall time:                      {elapsed / 3600:.2f} h")
if processed_this_session:
    log(f"  average:                        {elapsed / processed_this_session:.2f} s/volume")
    log(f"    of which download:            {dl_seconds / processed_this_session:.2f} s")
    log(f"    of which preprocess:          {pre_seconds / processed_this_session:.2f} s")
    log(f"    of which encode:              {enc_seconds / processed_this_session:.2f} s")
log(f"  OOM fallbacks:                  {oom_fallbacks}")
log("=" * 70)
log(f"Features: {REMOTE_ROOT}/train/ and {REMOTE_ROOT}/valid/")
log(f"Checkpoint: {REMOTE_CHECKPOINT}")
log(f"Log: {REMOTE_ROOT}/{LOG_FILE.name}")
